# 분석 기반 — 만족의 척도와 분석 표본

가설 검증에 앞서 **모든 가설이 함께 쓸 기준**을 확정.
고객 만족의 측정 대상, 분석 표본의 조건, 한 주문에 리뷰가 여럿일 때의 축약 규칙을 정하고
뷰 `order_review` 로 고정.

가설마다 기준을 따로 정하면 표본이 서로 달라져 결과를 나란히 놓을 수 없게 됨.

03 · 04 에서 정한 그레인과 조인 규칙, 표본에서 빠지는 행의 조건을 그대로 사용.

**확인 항목**

- 고객 만족의 정의 — 만족의 측정 대상
- 표본 — 사용할 주문과 제외되는 주문, 그 이유
- 중복 리뷰 — 한 주문에 달린 여러 점수을 하나로 줄이는 규칙
- 파생 지표 — 배송 소요 시간과 약속 대비의 정의
- 뷰 `order_review` — 위를 적용한 결과의 고정

## 0. 연결

04 에서 리뷰 텍스트의 빈 문자열을 `NULL` 로 바꾸었으므로 04 실행 이후의 상태가 전제.
처음부터 다시 돌릴 때는 02 로 원본을 복원한 뒤 03 · 04 · 05 순서로 실행.

In [1]:
import duckdb

con = duckdb.connect('../olist.duckdb')

## 1. 고객 만족의 정의

이 프로젝트에서 고객 만족을 무엇으로 측정할지에 대한 결정. 후보는 셋.

- **반복 주문** — 같은 사람의 주문이 데이터에 여러 건 남아 있는 경우
- **리뷰 점수** — 주문에 매겨진 1 ~ 5 의 점수
- **리뷰 본문** — 고객이 직접 쓴 문장

리뷰 본문은 후보에서 제외. 비어 있는 리뷰가 많고, 문장에서 만족을 판정하는 작업은 SQL 집계의
범위 밖이며, 결국 점수와 같은 리뷰에서 나오는 값이라 별개의 척도가 아님.
취소·반품도 후보가 되지 못함. 이 데이터에 반품 기록이 없고, 취소는 배송 전에 일어나는 일이라
배송을 겪은 뒤의 만족과는 무관.

남은 둘 중 반복 주문을 쓰려면 한 사람에게서 여러 건이 쌓여 있어야 함.
이 데이터셋이 Olist 거래 전량이라는 보장이 없어 한 사람의 주문이 빠짐없이 담겨 있다고 볼 수 없으므로,
재구매율 같은 고객 행동 지표로 부르지 않고 **같은 `customer_unique_id` 로 두 건 이상이 관측된 경우**를
`반복 주문` 으로 정의하고 그 관측량만 집계.

판단의 근거는 주문 횟수별 사람 수.

In [2]:
con.execute("""
WITH per_person AS (
    SELECT c.customer_unique_id,
           COUNT(*) AS order_cnt
    FROM orders    AS o
    JOIN customers AS c ON o.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
)
SELECT order_cnt,
       COUNT(*)                                                   AS people,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)         AS pct,
       SUM(order_cnt)                                             AS orders
FROM per_person
GROUP BY order_cnt
ORDER BY order_cnt
""").df()

,order_cnt,people,pct,orders
0,1,93099,96.88,93099.0
1,2,2745,2.86,5490.0
2,3,203,0.21,609.0
3,4,30,0.03,120.0
4,5,8,0.01,40.0
5,6,6,0.01,36.0
6,7,3,0.00,21.0
7,9,1,0.00,9.0
8,17,1,0.00,17.0


In [3]:
con.execute("""
WITH per_person AS (
    SELECT c.customer_unique_id,
           COUNT(*) AS order_cnt
    FROM orders    AS o
    JOIN customers AS c ON o.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
)
SELECT SUM(order_cnt)                                                AS orders,
       COUNT(*)                                                      AS people,
       COUNT(*) FILTER (order_cnt >= 2)                              AS people_2plus,
       ROUND(100.0 * COUNT(*) FILTER (order_cnt >= 2) / COUNT(*), 2) AS pct_2plus,
       COUNT(*) FILTER (order_cnt >= 4)                              AS people_4plus,
       MAX(order_cnt)                                                AS max_order_cnt
FROM per_person
""").df()

,orders,people,people_2plus,pct_2plus,people_4plus,max_order_cnt
0,99441.0,96096,2997,3.12,49,17


주문 99,441 건이 사람 96,096 명에게서 발생. 한 번만 주문한 사람이 93,099 명으로 96.88 %.
두 건 이상이 관측된 사람은 2,997 명(3.12 %) 이고 그중 대부분이 정확히 두 건.
네 건 이상은 49 명, 최다는 17 건 1 명.

반복 주문은 만족의 척도로 쓰지 않음. 사람의 96.88 % 에 대해 관측된 주문이 한 건뿐이라
그 사람의 만족 여부를 반복 주문에서 읽어낼 방법이 없음.

한 건뿐인 것을 불만족의 결과로 읽을 수도 없음. 그렇게 읽으면 고객의 96.88 % 가 불만족했다는
뜻이 되는데, 어떤 서비스에서도 나오기 어려운 비현실적인 수치.
반복 주문이 적다는 사실과 만족 여부는 다른 문제.

관측량이 적은 이유도 이 데이터로는 가릴 수 없음. 실제로 다시 사지 않은 것인지,
다시 살 시점이 수집 구간 밖인지, 애초에 그 주문이 데이터에 담기지 않은 것인지 구분할 근거가 없음.
이 수치는 재구매율이 아니라 관측량.

반복 주문을 만족의 신호로 읽으려면 두 번째 주문이 첫 주문을 **겪은 뒤에** 이루어졌는지도 확인이 필요.
장바구니가 쪼개져 같은 날 두 건이 되었거나 앞 주문이 도착하기 전에 다시 산 것이면 만족과 무관한 주문.
다만 관측량에서 이미 판정이 났으므로 그 확인은 생략.

**고객 만족은 리뷰 점수로 측정.** 주문마다 하나씩 붙고 1 ~ 5 를 벗어나는 값이 없어
주문 단위 분석에 그대로 사용 가능.

리뷰를 남긴 주문에만 있는 값이라는 한계는 존재. 점수가 달린 주문이 전체에서 얼마나 되는지는
2 절에서 확인.

## 2. 표본 정의

만족을 리뷰 점수로 측정하기로 했으므로, 쓸 수 있는 주문은 **점수가 달린 주문**.
여기에 수집 구간이 온전한 기간이라는 조건이 더해짐. 데이터셋의 양 끝은 달이 잘려 있어
어느 가설에서든 그 구간의 집계는 실제 규모를 나타내지 못함.

이 둘은 가설과 무관하게 걸리는 조건이므로 여기서 확정.
배송을 겪은 주문만 본다거나 결제 수단이 필요하다는 식의 조건은 가설마다 다르므로
각 가설 노트북에서 추가로 부여.

조건을 누적해 걸며 남는 행을 집계. 단계마다 줄어드는 폭이 그 조건에서 빠지는 주문의 규모.

- 2017-01 ~ 2018-08 — 수집 구간 양 끝의 잘린 달 제외
- 리뷰 있음 — 점수가 없는 주문 제외

리뷰 존재 여부의 확인에는 `EXISTS` 를 사용. 한 주문에 리뷰가 2 건 이상 달린 경우가 있어
`JOIN` 으로 세면 그런 주문이 두 번씩 계수됨.

In [4]:
con.execute("""
SELECT '1. 전체 주문' AS step,
       COUNT(*)      AS rows
FROM orders

UNION ALL
SELECT '2. + 온전한 기간(2017-01 ~ 2018-08)', COUNT(*)
FROM orders
WHERE order_purchase_timestamp >= DATE '2017-01-01'
  AND order_purchase_timestamp <  DATE '2018-09-01'

UNION ALL
SELECT '3. + 리뷰 있음', COUNT(*)
FROM orders AS o
WHERE o.order_purchase_timestamp >= DATE '2017-01-01'
  AND o.order_purchase_timestamp <  DATE '2018-09-01'
  AND EXISTS (SELECT 1 FROM order_reviews AS r WHERE r.order_id = o.order_id)

ORDER BY step
""").df()

,step,rows
0,1. 전체 주문,99441
1,2. + 온전한 기간(2017-01 ~ 2018-08),99092
2,3. + 리뷰 있음,98330


주문 99,441 건에서 시작해 표본은 98,330 건. 전체의 98.9 % 가 잔존.

- **기간 밖 349 건** — 수집 구간 양 끝의 잘린 달. 전체의 0.35 %
- **리뷰 없음 762 건** — 기간 안 주문 99,092 건의 0.77 %

두 조건 모두 빠지는 규모가 작음. 특히 리뷰가 없는 주문이 0.77 % 뿐이라,
만족을 리뷰 점수로 측정할 때 잃는 주문이 거의 없음.
1 절에서 남긴 한계 — 리뷰를 남긴 주문에만 값이 있다는 점 — 은 이 표본에서 크게 작용하지 않음.

가설에 따라 여기서 조건을 더 걸게 됨. 배송 소요 시간을 보는 가설이라면 배송이 완료되고
완료 시각이 남은 주문으로 좁히는 식.

## 3. 중복 리뷰를 하나로 줄이는 규칙

한 주문에 리뷰가 2 건 이상 달린 경우가 있어, `order_reviews` 를 그대로 붙이면 그런 주문이
여러 번 계수됨. 주문 하나에 점수 하나가 되도록 축약이 필요.

순서는 셋. 중복의 규모를 세고, 중복된 주문 안에서 점수이 어떻게 갈리는지 보고,
축약 규칙 후보들이 실제로 다른 결과를 내는지 비교.

먼저 표본 안에서 주문당 리뷰 건수의 분포를 집계.

In [5]:
con.execute("""
WITH sample AS (
    SELECT order_id
    FROM orders
    WHERE order_purchase_timestamp >= DATE '2017-01-01'
      AND order_purchase_timestamp <  DATE '2018-09-01'
),
per_order AS (
    SELECT s.order_id,
           COUNT(*) AS review_cnt
    FROM sample        AS s
    JOIN order_reviews AS r ON r.order_id = s.order_id
    GROUP BY s.order_id
)
SELECT review_cnt,
       COUNT(*)                                           AS orders,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct,
       SUM(review_cnt)                                    AS reviews
FROM per_order
GROUP BY review_cnt
ORDER BY review_cnt
""").df()

,review_cnt,orders,pct,reviews
0,1,97785,99.45,97785.0
1,2,541,0.55,1082.0
2,3,4,0.00,12.0


In [6]:
con.execute("""
WITH sample AS (
    SELECT order_id
    FROM orders
    WHERE order_purchase_timestamp >= DATE '2017-01-01'
      AND order_purchase_timestamp <  DATE '2018-09-01'
),
per_order AS (
    SELECT s.order_id,
           COUNT(*) AS review_cnt,
           arg_max(r.review_score, (r.review_creation_date, r.review_id))
         - arg_min(r.review_score, (r.review_creation_date, r.review_id)) AS score_gap
    FROM sample        AS s
    JOIN order_reviews AS r ON r.order_id = s.order_id
    GROUP BY s.order_id
)
SELECT score_gap,
       COUNT(*)                                           AS orders,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM per_order
WHERE review_cnt >= 2
GROUP BY score_gap
ORDER BY score_gap
""").df()

,score_gap,orders,pct
0,-4,26,4.77
1,-3,20,3.67
2,-2,23,4.22
3,-1,51,9.36
4,0,343,62.94
5,1,39,7.16
6,2,24,4.40
7,3,12,2.20
8,4,7,1.28


In [7]:
con.execute("""
WITH sample AS (
    SELECT order_id
    FROM orders
    WHERE order_purchase_timestamp >= DATE '2017-01-01'
      AND order_purchase_timestamp <  DATE '2018-09-01'
),
per_order AS (
    SELECT s.order_id,
           COUNT(*) AS review_cnt,
           arg_max(r.review_score, (r.review_creation_date, r.review_id)) AS last_score,
           arg_min(r.review_score, (r.review_creation_date, r.review_id)) AS first_score,
           AVG(r.review_score)                                            AS avg_score,
           MIN(r.review_score)                                            AS min_score,
           MAX(r.review_score)                                            AS max_score
    FROM sample        AS s
    JOIN order_reviews AS r ON r.order_id = s.order_id
    GROUP BY s.order_id
),
rules AS (
    SELECT '1. 마지막' AS rule, last_score  AS score, review_cnt FROM per_order
    UNION ALL SELECT '2. 최초',  first_score, review_cnt FROM per_order
    UNION ALL SELECT '3. 평균',  avg_score,   review_cnt FROM per_order
    UNION ALL SELECT '4. 최저',  min_score,   review_cnt FROM per_order
    UNION ALL SELECT '5. 최고',  max_score,   review_cnt FROM per_order
)
SELECT rule,
       ROUND(AVG(score), 4)                          AS avg_all,
       ROUND(AVG(score) FILTER (review_cnt >= 2), 3) AS avg_dup,
       COUNT(*) FILTER (score <= 2)                  AS low_orders,
       COUNT(*) FILTER (score >= 4)                  AS high_orders
FROM rules
GROUP BY rule
ORDER BY rule
""").df()

,rule,avg_all,avg_dup,low_orders,high_orders
0,1. 마지막,4.0885,3.919,14383,75835
1,2. 최초,4.0896,4.121,14363,75871
2,3. 평균,4.0890,4.020,14337,75820
3,4. 최저,4.0869,3.642,14421,75794
4,5. 최고,4.0911,4.398,14325,75912


표본 98,330 건 중 리뷰가 하나인 주문이 97,785 건으로 99.45 %. 둘인 주문 541 건,
셋인 주문 4 건으로 중복은 545 건.

중복 545 건에서 나중 리뷰와 첫 리뷰의 점수 차이는 0 이 343 건(62.94 %).
나머지 202 건 중 나중이 더 낮은 경우가 120 건, 더 높은 경우가 82 건으로 낮아지는 쪽이 1.5 배.
차이가 −4 인 주문이 26 건으로, 5 점을 남긴 뒤 1 점을 다시 남긴 경우.

축약 규칙 후보 다섯을 같은 표본에 적용한 결과, 전체 평균은 4.0869 ~ 4.0911 로 폭이 0.004.
2 점 이하 주문은 14,325 ~ 14,421 건(폭 96 건), 4 점 이상 주문은 75,794 ~ 75,912 건(폭 118 건).
어느 규칙을 써도 집계 결과가 달라지지 않음.

규칙이 갈리는 것은 중복 545 건 안에서뿐. 그 545 건의 평균은 최저 3.642 에서 최고 4.398 까지
0.76 점 차이. 전체 98,330 건에 대한 545 건의 비중이 0.55 % 라 그 차이가 묻힘.

**규칙 — 주문마다 마지막에 작성된 리뷰 하나만 유지.** 결과가 사실상 같은 이상 근거가 분명한 쪽을
택함. 점수이 갈릴 때는 나중의 평가가 그 주문에 대한 최종 판단.
평균을 내면 1 ~ 5 의 정수라는 성질이 깨져 구간별 집계에서 취급이 번거로움.
최저·최고는 한쪽 극단만 남겨 갈린 평가를 대표하지 못함.
작성 시각이 같은 경우에는 `review_id` 순으로 하나를 골라 실행할 때마다 동일한 결과를 보장.

마지막을 택하면 최초를 택할 때보다 전체 평균이 0.0011 낮음. 나중 리뷰가 점수를 내리는 쪽이
더 많기 때문이며, 그 차이는 집계에 드러나지 않는 규모.

규칙을 적용한 결과는 뷰 `order_review`. 표본 조건과 축약 규칙을 노트북마다 다시 쓰지 않기 위한 것이며,
주(州)·도시와 배송 관련 시각, 주문 상태도 함께 담아 가설 노트북에서 조건을 더 걸 수 있도록 구성.

In [8]:
con.execute("""
CREATE OR REPLACE VIEW order_review AS
SELECT o.order_id,
       c.customer_unique_id,
       c.customer_state,
       c.customer_city,
       o.order_status,
       o.order_purchase_timestamp,
       o.order_delivered_customer_date,
       o.order_estimated_delivery_date,
       date_diff('day', o.order_purchase_timestamp,
                        o.order_delivered_customer_date) AS delivery_days,
       date_diff('day', o.order_estimated_delivery_date,
                        o.order_delivered_customer_date) AS days_vs_promise,
       r.review_id,
       r.review_score,
       r.review_creation_date
FROM orders        AS o
JOIN customers     AS c ON o.customer_id = c.customer_id
JOIN order_reviews AS r ON o.order_id    = r.order_id
WHERE o.order_purchase_timestamp >= DATE '2017-01-01'
  AND o.order_purchase_timestamp <  DATE '2018-09-01'
QUALIFY ROW_NUMBER() OVER (PARTITION BY o.order_id
                           ORDER BY r.review_creation_date DESC, r.review_id DESC) = 1
""")

con.execute("""
SELECT COUNT(*)                    AS rows,
       COUNT(DISTINCT order_id)    AS orders,
       COUNT(DISTINCT review_id)   AS reviews,
       MIN(review_score)           AS min_score,
       MAX(review_score)           AS max_score
FROM order_review
""").df()

,rows,orders,reviews,min_score,max_score
0,98330,98330,97767,1,5


In [9]:
con.execute("""
WITH per_review AS (
    SELECT review_id,
           COUNT(*) AS order_cnt
    FROM order_review
    GROUP BY review_id
)
SELECT order_cnt,
       COUNT(*)        AS reviews,
       SUM(order_cnt)  AS orders
FROM per_review
GROUP BY order_cnt
ORDER BY order_cnt
""").df()

,order_cnt,reviews,orders
0,1,97216,97216.0
1,2,539,1078.0
2,3,12,36.0


뷰의 행 수와 주문 수가 98,330 으로 동일. 축약이 의도대로 되어 주문 하나에 점수 하나.
점수은 1 ~ 5 안에 분포.

다만 `review_id` 는 97,767 개로 주문 수보다 563 개 적음. 하나의 리뷰가 여러 주문에 걸려 있다는 뜻이라
방향을 바꿔 확인. 리뷰 하나가 주문 둘에 걸린 경우가 539 건, 셋에 걸린 경우가 12 건으로,
1,114 개 주문(1.13 %)이 다른 주문과 리뷰를 공유.

한 번의 구매가 여러 주문으로 나뉘어 기록되고 리뷰는 한 번만 달린 경우로 추정.
주문 단위로 집계하면 그 주문들이 같은 점수을 공유. 서로 독립적인 관측이 아니라는 뜻이지만
1 % 규모라 집계에 영향이 없어 그대로 유지.

## 4. 파생 지표의 정의

뷰에 함께 담은 두 값. 배송을 다루는 가설이면 공통으로 쓰게 되므로 여기서 정의.

**배송 소요 시간** `delivery_days` — 구매일과 수령일의 **날짜 차이**.
경과 시간을 시 단위로 재지 않는 이유는 고객이 세는 방식이 날짜이기 때문.
화요일에 주문해 목요일에 받으면 이틀 걸린 것이지 43 시간 걸린 것이 아님.
만족도를 설명하려는 분석이므로 고객이 체감하는 단위로 계산.

**약속 대비** `days_vs_promise` — 예상 도착일과 실제 수령일의 날짜 차이. 양수면 늦은 것.
예상 도착일은 시각 없이 날짜만 기록되어 있어 날짜 비교가 정확하며,
`delivery_days` 와 단위가 같아 서로 대조 가능.

배송이 완료되지 않은 주문은 수령일이 없어 두 값 모두 `NULL`. 그 규모를 확인.

In [10]:
con.execute("""
SELECT COUNT(*)                        AS rows,
       COUNT(delivery_days)            AS with_days,
       COUNT(*) - COUNT(delivery_days) AS null_days,
       MIN(delivery_days)              AS min_days,
       MAX(delivery_days)              AS max_days,
       MIN(days_vs_promise)            AS min_vs_promise,
       MAX(days_vs_promise)            AS max_vs_promise
FROM order_review
""").df()

,rows,with_days,null_days,min_days,max_days,min_vs_promise,max_vs_promise
0,98330,95561,2769,0,208,-147,188


## 5. 정리

이후 가설 노트북이 전제로 삼을 기준.

- **고객 만족은 리뷰 점수(1 ~ 5)로 측정.** 반복 주문은 사람의 96.88 % 에 대해 주문이 한 건만
  관측되어 척도로 사용 불가
- **공용 표본은 98,330 건.** 2017-01 ~ 2018-08 · 리뷰 있음. 가설별 조건은 각 노트북에서 추가
- **주문 하나에 점수 하나.** 리뷰가 여럿인 주문 545 건은 마지막에 작성된 것만 유지
- **뷰 `order_review`** 에 위 셋을 적용. 주문 식별자, 사람 식별자, 주(州)·도시, 주문 상태,
  구매 · 수령 · 배송 예정 시각, 점수을 담고 있어 가설 노트북은 여기서 조건을 더 걸어 집계
- 리뷰 하나가 여러 주문에 걸린 1,114 개 주문은 서로 독립적인 관측이 아님. 1 % 규모

가설마다 표본이 달라지더라도 만족의 정의와 축약 규칙은 유지.
변경이 필요하면 05 를 고치고 이후 노트북을 다시 실행.

## 6. 연결 종료

DuckDB 파일은 한 프로세스만 쓰기 모드로 열 수 있으므로 다음 노트북을 위해 여기서 종료.
뷰 `order_review` 는 DB 파일에 저장되어 연결을 닫아도 유지.

In [11]:
con.close()